# CIFAR-10 Autoencoder — Analysis

Training now happens in `train.py`:

    uv run python projects/cifar10/autoencoder/train.py

This notebook only loads the resulting checkpoint for analysis:
training curves and reconstruction quality.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torchmetrics.functional.image import (
    peak_signal_noise_ratio,
    structural_similarity_index_measure,
)
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

from chimera.data import CIFAR10DataModule
from chimera.models import CIFARAutoencoder

RUN_DIR = "/mnt/ai/runs/cifar10/autoencoder"
DATA_DIR = "/mnt/ai/data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_DIM = 128

## Load checkpoint

In [ ]:
ckpt = torch.load(
    f"{RUN_DIR}/checkpoints/ae.ckpt", map_location="cpu", weights_only=False
)
model_state = {
    k.removeprefix("model."): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("model.")
}

model = CIFARAutoencoder(in_channels=3, latent_dim=LATENT_DIM)
model.load_state_dict(model_state)
model.to(DEVICE).eval()
print(
    f"loaded AE ({sum(p.numel() for p in model.parameters()):,} params, epoch {ckpt['epoch']})"
)

## Data

Load CIFAR-10. Pixels stay in `[0, 1]`, matching the decoder's sigmoid output.

In [ ]:
dm = CIFAR10DataModule(data_dir=DATA_DIR, pin_memory=False, num_workers=0)
dm.prepare_data()
dm.setup("test")
test_loader = dm.test_dataloader()

In [ ]:
images, _ = next(iter(test_loader))

grid = make_grid(images[:8].clamp(0, 1), nrow=8)
plt.figure(figsize=(12, 2))
plt.imshow(to_pil_image(grid))
plt.title("Sample CIFAR-10 Images")
plt.axis("off")
plt.show()

## Training curves

In [ ]:
csv_path = sorted(glob.glob(f"{RUN_DIR}/csv/version_*/metrics.csv"))[-1]
metrics = pd.read_csv(csv_path)

plt.figure(figsize=(7, 5))
plt.title("Reconstruction Loss")
for stage in ["train", "val"]:
    col = f"{stage}/loss"
    d = metrics.dropna(subset=[col])
    plt.plot(d["step"], d[col], marker="o" if stage == "val" else None, label=stage)
plt.xlabel("Step")
plt.ylabel("MAE")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Reconstruction quality

In [ ]:
@torch.no_grad()
def recon_metrics(model, loader, device):
    maes, psnrs, ssims = [], [], []
    for x, _ in loader:
        x = x.to(device)
        recon = model(x)
        maes.append(F.l1_loss(recon, x).item())
        psnrs.append(peak_signal_noise_ratio(recon, x, data_range=1.0).item())
        ssims.append(
            structural_similarity_index_measure(recon, x, data_range=1.0).item()
        )
    return np.mean(maes), np.mean(psnrs), np.mean(ssims)


mae, psnr, ssim = recon_metrics(model, test_loader, DEVICE)
print(f"test MAE {mae:.4f}, PSNR {psnr:.2f} dB, SSIM {ssim:.3f}")

images, _ = next(iter(test_loader))
with torch.no_grad():
    recons = model(images.to(DEVICE)).float().cpu()

n = 8
# Row 0: originals, row 1: reconstructions.
batch = torch.cat([images[:n], recons[:n]]).clamp(0, 1)
grid = make_grid(batch, nrow=n)
plt.figure(figsize=(12, 3))
plt.imshow(to_pil_image(grid))
plt.title("Original (top) vs Reconstruction (bottom)")
plt.axis("off")
plt.show()